# Lending fairness audit — replication on a synthetic fixture

Runs `fairscope`'s `LendingFairnessAudit` on the **synthetic** fixture committed in the repo (`tests/fixtures/lending_subsample.csv`). The fixture is *not* real HMDA data; it is seed-generated to reproduce the **direction and approximate magnitude** of the mortgage-fairness findings (minority approval rate < reference, gap widening; heterogeneous effect where `x0 > 0`). It demonstrates the audit pipeline, not an empirical result.

In [ ]:
import matplotlib

matplotlib.use('Agg')
from pathlib import Path

import numpy as np
import pandas as pd

from fairscope.lending import LendingFairnessAudit

In [ ]:
fixture = Path('..') / 'tests' / 'fixtures' / 'lending_subsample.csv'
if not fixture.exists():
    fixture = Path('tests') / 'fixtures' / 'lending_subsample.csv'
df = pd.read_csv(fixture, comment='#')
df.head()

## Annual approval-gap analysis (descriptive)

Per year and group: approval rate and symmetric disparate impact versus the reference group. This is descriptive — no causal claim.

In [ ]:
report = LendingFairnessAudit.from_outcomes(
    df['approved'].to_numpy(),
    df['group'].to_numpy(),
    df['year'].to_numpy(),
    reference='reference',
).run()
report.to_dataframe()

In [ ]:
print(report.summary())

The minority disparate impact falls below the four-fifths threshold and **widens** across years — the planted direction and magnitude.

## Subgroup CATE (optional, `fairscope[lending]`)

`estimate_cate` wraps `econml.dml.CausalForestDML` (the estimator used in the lending paper). The causal claim is **conditional on the DML assumptions** (unconfoundedness given the supplied features, overlap); it does not, on its own, prove discrimination. If `econml` is not installed the cell prints how to enable it.

In [ ]:
audit = LendingFairnessAudit.from_outcomes(
    df['approved'].to_numpy(), df['group'].to_numpy(), df['year'].to_numpy(),
    reference='reference',
)
X = df[['x0', 'x1', 'x2']].to_numpy()
try:
    res = audit.estimate_cate(X, n_estimators=200)
    strong = X[:, 0] > 0
    print('ATE:', round(res['ate'], 3))
    print('median CATE, x0>0 :', round(float(np.median(res['effect'][strong])), 3))
    print('median CATE, x0<=0:', round(float(np.median(res['effect'][~strong])), 3))
except ImportError as exc:
    print('CATE skipped:', exc)

The effect is negative overall (minority status lowers approval) and **more negative on the `x0 > 0` stratum** — the planted heterogeneity.

## The paper's other strategies: DiD and RDD

The lending papers also use difference-in-differences (DiD) and regression-discontinuity (RDD). These are **paper-specific identification strategies, not `fairscope` primitives** (one paper's appendix RDD/DiD also failed its validity tests). Below is a **purely illustrative** DiD on this synthetic data — treat = minority, post = year ≥ 2020 — using `statsmodels`. It is not a library function and carries no validity guarantee. RDD needs a running variable (e.g. LTV at an 80% cutoff), which this fixture does not include; see the paper.

In [ ]:
import statsmodels.formula.api as smf

ld = df.copy()
ld['treat'] = (ld['group'] != 'reference').astype(int)
ld['post'] = (ld['year'] >= 2020).astype(int)
did = smf.ols('approved ~ treat + post + treat:post', data=ld).fit()
print('DiD (treat:post) coefficient:', round(did.params['treat:post'], 3))
print('  illustrative only — NOT a fairscope primitive, no validity guarantee')